# 02 — Preprocessing

Applies bandpass filtering, EOG channel labeling, standard montage, and resampling.
All parameters are read from the YAML config — no code changes needed between experiments.

**Input:** `<subject>_raw.fif`  
**Output:** `<subject>_preprocessed_raw.fif`, `<subject>_preprocessed_events_eve.fif`

In [ ]:
%load_ext autoreload
%autoreload 2

from eeg_toolkit import load_config, find_subjects, preprocess_subject

# ── Update this path to point to your experiment config ──
cfg = load_config('../configs/your_experiment.yaml')

subjects = find_subjects(cfg)
print(f"Subjects to preprocess: {len(subjects)}")
print(subjects)

In [ ]:
# ── Test with the first subject ──
test_subject = subjects[0]
print(f"\nTesting preprocessing on: {test_subject}\n")

success = preprocess_subject(cfg, test_subject, overwrite=False, verbose=True)
print(f"\nResult: {'OK' if success else 'skipped or failed'}")

In [ ]:
# ── Verify preprocessed output ──
import mne
from eeg_toolkit import get_subject_path

raw_pp = mne.io.read_raw_fif(
    get_subject_path(cfg, test_subject, 'preprocessed_raw'),
    preload=False, verbose='WARNING'
)
events_pp = mne.read_events(
    get_subject_path(cfg, test_subject, 'preprocessed_events')
)

print(f"Channels:     {len(raw_pp.ch_names)}")
print(f"Sample rate:  {raw_pp.info['sfreq']} Hz")
print(f"Duration:     {raw_pp.times[-1]:.1f} s")
print(f"Channel types: {set(raw_pp.get_channel_types())}")
print(f"Has montage:  {raw_pp.info['dig'] is not None}")
print(f"\nEvents: {len(events_pp)}")

In [ ]:
# ── Run preprocessing for all subjects ──
from eeg_toolkit import preprocess_all

# The first subject will be skipped (already done above with overwrite=False).
# Expect ~30–60 s per subject.
summary = preprocess_all(cfg, overwrite=False, verbose=True)